# 3. Sample model

A sample model is a structure in its buffer: a solvent, and maybe a salt. Here we build an asymmetric DOPC/POPC vesicle in TIP3 water with 0.15 mol/L NaCl.

In this notebook you will:

1. load the components from the library
2. set up the vesicle and its leaflets
3. set up the buffer
4. assemble the sample model, which runs GROMACS
5. look at the model and export it

Run the setup cell first. On Colab it installs GROMACS and Shapespyer, which takes about 3 minutes.

In [ ]:
# @title Setup: on Colab, install GROMACS and Shapespyer (about 3 min)
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
CONFIG_FILE = Path("shapespyer.yaml")  # where Shapespyer keeps its files

if IN_COLAB:
    !apt-get -qq update > /dev/null && apt-get -qq install -y --no-install-recommends gromacs > /dev/null
    %pip install -q "shapespyer==0.3.0"

    # nglview, behind .show(), draws molecules through Colab's widget manager
    from google.colab import output
    output.enable_custom_widget_manager()

    # the library, Temp and Projects go to /content, to browse them in the Files panel
    from shapes.project import Config
    config = Config(CONFIG_FILE)
    config.app_path = "/content/shapespyer"
    config.save()
elif not CONFIG_FILE.is_file():
    err_msg = f"No {CONFIG_FILE} in {Path.cwd()}; open the notebook from the shapespyer-workshop folder"
    raise FileNotFoundError(err_msg)

from shapes.project.files import AppFileManager

# the first call sets where the files go, for the whole session
config = AppFileManager(CONFIG_FILE).config
print("Shapespyer files:", config.app_path)
if config.gmx_problem is not None:
    err_msg = (
        f"This notebook needs GROMACS: {config.gmx_problem}. Install it, or set gmx_executable"
        f" in {CONFIG_FILE}. On Windows, the easy way is to run this notebook in WSL or on Colab."
    )
    raise RuntimeError(err_msg)
print("GROMACS:", config.gmx_version)

In [ ]:
import shutil
from pathlib import Path

import mdtraj
import nglview

from shapes.basics.utils import LogConfiguration
from shapes.project import Ion, Library, Lipid, SampleModel, Vesicle
from shapes.project.files import AppFileManager
from shapes.project.species import Buffer, Salt, Solvent

LogConfiguration("WARNING")

library = Library()
library.clean_temp()
EXPORTS = Path("model_exports").resolve()

### Load the components

In [ ]:
dopc = Lipid.load("DOPC")
popc = Lipid.load("POPC")
dopc.show()

In [ ]:
popc.show()

### Set up the vesicle

A component added to the structure joins every layer.

In [ ]:
vesicle = Vesicle(name="DemoVesicle")
vesicle.components.append(dopc)
vesicle.components.append(popc)
vesicle.fill = "FIBO"

print(vesicle)

### One lamella with different leaflets

- inner leaflet: POPC and DOPC in the ratio 2 : 3
- outer leaflet: DOPC only (POPC is marked absent)

In [ ]:
lamella = vesicle.lamellae[0]
lamella.symmetric = False
lamella.rmin = 3

lamella.inner_leaflet_composition.set_mol_ratio("POPC", 2)
lamella.inner_leaflet_composition.set_mol_ratio("DOPC", 3)
lamella.inner_leaflet_dmin = 1.1

lamella.outer_leaflet_composition.set_presence("POPC", False)
lamella.outer_leaflet_composition.set_mol_ratio("DOPC", 100)
lamella.outer_leaflet_dmin = 2

print(vesicle.lamellae)

### Set up the buffer

TIP3 water with 0.15 mol/L NaCl. DOPC and POPC are neutral, so no counter ions are needed.

In [ ]:
buffer = Buffer(name="DemoBuffer", solvent=Solvent.load("TIP3"))
buffer.components.append(
    Salt(name="NaCl", cation=Ion.load("NAT"), anion=Ion.load("CLA"), concentration=0.15)
)

model = SampleModel(vesicle, buffer=buffer, name="DemoModel")
print(model)

### Assemble the sample model

`.assemble()` builds the vesicle, puts in the salt ions with `gmx insert-molecules` and fills the box with water with `gmx solvate`. Then the model holds its `.gro`, its `.top` and `assembly.log`.

If you change any setting of the model or the vesicle, call `.assemble()` again.

In [ ]:
model.assemble()
model.paths

The job of the assembly keeps a log of every command it ran, with the output. These are the GROMACS commands:

In [ ]:
print("\n".join(line for line in model.job.log.read_text().splitlines() if line.startswith("$ ")))

### Look at the model

We hide the water, so the vesicle and the ions are easy to see.

In [ ]:
model_gro = next(path for path in model.paths if path.suffix == ".gro")
frame = mdtraj.load(model_gro)
print(frame.n_atoms, "atoms, box", frame.unitcell_lengths[0], "nm")
# mdtraj renames TIP3 to HOH on loading, so select it with the "water" keyword
shown = frame.atom_slice(frame.top.select("not water"))
print(shown.n_atoms, "atoms without the water")
nglview.show_mdtraj(shown)

### Export and download

`export_to()` copies the files of the model, and of every asset in it, into a folder you choose. The vesicle is assembled on its own too, so the export also holds the vesicle's own `.gro`.

In [ ]:
vesicle.assemble()
model.export_to(EXPORTS / "DemoModel", overwrite=True)

for path in sorted(EXPORTS.rglob("*")):
    if path.is_file():
        print(path.relative_to(EXPORTS))

In [ ]:
archive = shutil.make_archive("model_exports", "zip", EXPORTS)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
archive

### Clean up (optional)

This deletes the exports, the zip and Temp, where the vesicle and the model were assembled. `force=True` is needed for Temp, because the assets of this session still hold their files there.

In [ ]:
AppFileManager().delete_dir(EXPORTS, force=True)
Path(archive).unlink(missing_ok=True)
library.clean_temp(force=True)
print(EXPORTS.exists())